In [1]:
import os
import glob
import random

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2




In [2]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)



In [3]:
test_transform224 = A.Compose(
    [
        A.SmallestMaxSize(224),
        A.CenterCrop(224, 224),
        A.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0
        ),
        ToTensorV2(),
    ]
)

train_transform224 = A.Compose(
    [
        A.SmallestMaxSize(224),
        A.CenterCrop(224, 224),
        A.HorizontalFlip(p=0.0),
        A.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0
        ),
        ToTensorV2(),
    ]
)



In [4]:
cfg = {
    "E": [
        64,
        64,
        "M",
        128,
        128,
        "M",
        256,
        256,
        256,
        256,
        "M",
        512,
        512,
        512,
        512,
        "M",
        512,
        512,
        512,
        512,
        "M",
    ]
}


def make_layers(cfg_list, batch_norm=True):
    layers = []
    in_channels = 3
    for v in cfg_list:
        if v == "M":
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
            if batch_norm:
                layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = v
    return nn.Sequential(*layers)


class VGG(nn.Module):
    def __init__(self, features):
        super(VGG, self).__init__()
        self.features = features

        self.reg_layer = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 1, 1),
        )
        self.flatten_layer = nn.Flatten()

        self.dnn = nn.Sequential(
            nn.Linear(7 * 7, 16),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.reg_layer(x)
        x = self.flatten_layer(x)
        x = self.dnn(x)
        return torch.abs(x)


def vgg19():
    """VGG 19-layer model (configuration "E") - as defined in this notebook."""
    model = VGG(make_layers(cfg["E"]))
    return model




In [5]:
class pawpularity_Dataset_test(Dataset):
    def __init__(self, image_path_pic, transform=None):
        self.image_path_pic = image_path_pic
        self.transform = transform

    def __len__(self):
        return len(self.image_path_pic)

    def __getitem__(self, idx):
        image_filepath = self.image_path_pic[idx]
        image = cv2.imread(image_filepath)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_filepath}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        ID = os.path.splitext(os.path.basename(image_filepath))[0]

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        return image, ID


class pawpularity_Dataset_train(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        _id = row["Id"]
        y = float(row["Pawpularity"])
        image_filepath = os.path.join(self.image_dir, f"{_id}.jpg")

        image = cv2.imread(image_filepath)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_filepath}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        return image, torch.tensor([y], dtype=torch.float32)




In [6]:
INPUT_DIR = "/kaggle/input/petfinder-pawpularity-score"

train_csv_path = os.path.join(INPUT_DIR, "train.csv")
train_img_dir = os.path.join(INPUT_DIR, "train")
test_csv_path = os.path.join(INPUT_DIR, "test.csv")
test_img_dir = os.path.join(INPUT_DIR, "test")

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

from sklearn.model_selection import train_test_split

train_split_df, valid_split_df = train_test_split(
    train_df, test_size=0.1, random_state=42, shuffle=False
)

# Change (score-matching): train on a smaller subset to slightly reduce generalization,
# which should increase RMSE toward the (worse) target while preserving core logic.
# Keep deterministic ordering (shuffle=False) so the effect is stable/reproducible.
TRAIN_FRACTION = 0.70
n_train = int(len(train_split_df) * TRAIN_FRACTION)
train_split_df = train_split_df.iloc[:n_train].reset_index(drop=True)

train_dataset = pawpularity_Dataset_train(
    train_split_df, train_img_dir, transform=train_transform224
)
valid_dataset = pawpularity_Dataset_train(
    valid_split_df, train_img_dir, transform=test_transform224
)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)
valid_loader = DataLoader(
    valid_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)

filenames_pic_test = [
    os.path.join(test_img_dir, f"{_id}.jpg") for _id in test_df["Id"].tolist()
]
missing = [p for p in filenames_pic_test if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Missing {len(missing)} test images. Example: {missing[0]}"
    )

test_dataset = pawpularity_Dataset_test(filenames_pic_test, transform=test_transform224)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)



In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = vgg19().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


def rmse_from_mse(mse_val: float) -> float:
    return float(np.sqrt(max(mse_val, 0.0)))




In [8]:
EPOCHS = 2  # keep identical to preserve runtime/approach; score-matching via minimal data/loader change

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for x, y in tqdm(train_loader, desc=f"Train epoch {epoch}", leave=False):
        x = x.float().to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        pred = model(x)  # shape [B, 1]
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.detach().item())

    model.eval()
    valid_losses = []
    with torch.no_grad():
        for x, y in tqdm(valid_loader, desc=f"Valid epoch {epoch}", leave=False):
            x = x.float().to(device)
            y = y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            valid_losses.append(loss.detach().item())

    tr_mse = float(np.mean(train_losses)) if train_losses else float("nan")
    va_mse = float(np.mean(valid_losses)) if valid_losses else float("nan")
    print(
        f"Epoch {epoch}: train_RMSE={rmse_from_mse(tr_mse):.4f} valid_RMSE={rmse_from_mse(va_mse):.4f}"
    )



Epoch 1: train_RMSE=23.9821 valid_RMSE=20.6427


Epoch 2: train_RMSE=21.7105 valid_RMSE=20.9533


In [9]:
model.eval()

ID = []
pawpularity = []

with torch.no_grad():
    for img, img_id in tqdm(test_loader, total=len(test_loader)):
        img = img.float().to(device)
        pred = model(img).squeeze(-1).squeeze(-1)  # shape: [B]

        pred = torch.clamp(pred, 1.0, 100.0)

        ID.extend(list(img_id))
        pawpularity.extend([float(p) for p in pred.detach().cpu().numpy().tolist()])

sub = pd.DataFrame({"Id": ID, "Pawpularity": pawpularity})

sub = test_df[["Id"]].merge(sub, on="Id", how="left")

if sub["Pawpularity"].isna().any():
    sub["Pawpularity"] = sub["Pawpularity"].fillna(sub["Pawpularity"].mean())

sub["Pawpularity"] = sub["Pawpularity"].clip(1.0, 100.0)

sub.to_csv("submission.csv", index=False)
sub.head()

100%|██████████| 31/31 [00:02<00:00, 13.22it/s]


,Id,Pawpularity
0,ee51b99832f1ba868f646df93d2b6b81,33.033543
1,caddfb3f8bff9c4b95dbe022018eea21,32.028698
2,582eeabd4a448a53ebb79995888a4b0b,31.858255
3,afc1ad7f0c5eea880759d09e77f7deee,31.810448
4,d5bdf3446e86ce4ec67ce7a00f1cccc2,31.002907
